# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## 1. Setup & Imports

In [11]:
!nvidia-smi

Wed Mar 25 11:53:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:1A:00.0 Off |                  N/A |
| 29%   23C    P8              1W /  250W |     502MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
%pip install numpy
%pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [13]:
import warnings
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## 2. Hyperparameters

In [14]:
BATCH_SIZE = 256
NUM_EPOCHS = 10
LEARNING_RATE = 1e-2
NUM_WORKERS = 8

## 3. Data Loading

In [15]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

classes: list[str] = train_set.classes
print("Classes:", classes)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## 4. Model Definition

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [16]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(self, num_classes: int = 10, activation: nn.Module = nn.LeakyReLU()) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(3,  32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.act  = activation
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.act(self.fc1(x))
        return self.fc2(x)

## 5. Training & Evaluation Helpers

`train_one_epoch` and `validate` each handle a single pass through their respective loader.  
`fit` orchestrates the loop, tracks history, and restores the best checkpoint.

In [21]:
criterion = nn.CrossEntropyLoss()

def train(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> tuple[float, float]:
    """Run one full pass over `loader` in training mode.

    Returns
    -------
    train_loss : float  – mean cross-entropy loss over all samples
    train_acc  : float  – accuracy in percent
    """
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct      += (outputs.argmax(dim=1) == labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def validate(
    model: nn.Module,
    loader: DataLoader,
) -> tuple[float, float]:
    """Run one full pass over `loader` in evaluation mode.

    Returns
    -------
    val_loss : float  – mean cross-entropy loss over all samples
    val_acc  : float  – accuracy in percent
    """
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            correct      += (outputs.argmax(dim=1) == labels).sum().item()
            total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def fit(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int = NUM_EPOCHS,
) -> dict[str, list[float]]:
    """Train `model` for `num_epochs`, validating after every epoch.

    Saves the weights that achieved the best validation accuracy and
    restores them at the end.

    Returns
    -------
    history : dict with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """
    best_val_acc = 0.0
    best_state   = None
    history: dict[str, list[float]] = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
    }

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train(model, train_loader, optimizer)
        val_loss, val_acc = validate(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:>{len(str(num_epochs))}}/{num_epochs} | "
            f"train loss {train_loss:.4f}, train acc {train_acc:.2f}% | "
            f"val loss {val_loss:.4f}, val acc {val_acc:.2f}%"
        )

        wandb.log({
            "train/loss": train_loss,
            "train/accuracy": train_acc,
            "val/loss": val_loss,
            "val/accuracy": val_acc,
            "epoch": epoch
        })

    # Restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best weights (val acc {best_val_acc:.2f}%)")

    return history

## 6. Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [22]:
wandb.finish()

wandb.init(
    entity="d7047e-group12",
    project="Lab0",
    name="Experiment A – SGD + LeakyReLU",
    config={
        "model": "SimpleCNN",
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "optimizer": "Adam",
        # Add any other hyperparameters you want to track
    }
)

In [23]:
model_a    = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=1e-2)

print("=== Experiment A: SGD + LeakyReLU ===")
history_a = fit(model_a, optimizer_a, train_loader, test_loader)

=== Experiment A: SGD + LeakyReLU ===
Epoch  1/10 | train loss 2.2985, train acc 10.17% | val loss 2.2922, val acc 11.84%
Epoch  2/10 | train loss 2.2807, train acc 14.96% | val loss 2.2607, val acc 17.16%
Epoch  3/10 | train loss 2.2166, train acc 21.20% | val loss 2.1499, val acc 24.83%
Epoch  4/10 | train loss 2.0857, train acc 25.57% | val loss 2.0148, val acc 27.51%
Epoch  5/10 | train loss 1.9792, train acc 28.67% | val loss 1.9253, val acc 30.93%
Epoch  6/10 | train loss 1.8977, train acc 32.07% | val loss 1.8502, val acc 33.73%
Epoch  7/10 | train loss 1.8231, train acc 34.80% | val loss 1.7816, val acc 36.36%
Epoch  8/10 | train loss 1.7486, train acc 37.14% | val loss 1.7454, val acc 35.96%
Epoch  9/10 | train loss 1.6839, train acc 39.21% | val loss 1.7333, val acc 38.22%
Epoch 10/10 | train loss 1.6348, train acc 41.02% | val loss 1.6009, val acc 41.16%

Restored best weights (val acc 41.16%)


epoch,▁▂▃▃▄▅▆▆▇█
train/accuracy,▁▂▄▄▅▆▇▇██
train/loss,██▇▆▅▄▃▂▂▁
val/accuracy,▁▂▄▅▆▆▇▇▇█
val/loss,██▇▅▄▄▃▂▂▁
epoch,10
train/accuracy,41.018
train/loss,1.63485
val/accuracy,41.16
val/loss,1.60091


## 7. Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [9]:
model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=1e-3)

print("=== Experiment B: Adam + LeakyReLU ===")
history_b = fit(model_b, optimizer_b, train_loader, test_loader)

=== Experiment B: Adam + LeakyReLU ===
Epoch  1/10 | train loss 1.5373, train acc 44.19% | val loss 1.2321, val acc 55.77%
Epoch  2/10 | train loss 1.1415, train acc 59.28% | val loss 1.0538, val acc 62.76%
Epoch  3/10 | train loss 0.9466, train acc 66.76% | val loss 0.9465, val acc 67.20%
Epoch  4/10 | train loss 0.8095, train acc 71.76% | val loss 0.8256, val acc 71.28%
Epoch  5/10 | train loss 0.7131, train acc 75.12% | val loss 0.8082, val acc 72.11%
Epoch  6/10 | train loss 0.6175, train acc 78.52% | val loss 0.7715, val acc 73.20%
Epoch  7/10 | train loss 0.5415, train acc 80.99% | val loss 0.7333, val acc 74.97%
Epoch  8/10 | train loss 0.4693, train acc 83.79% | val loss 0.7218, val acc 75.60%
Epoch  9/10 | train loss 0.4057, train acc 85.89% | val loss 0.7530, val acc 75.46%
Epoch 10/10 | train loss 0.3402, train acc 88.25% | val loss 0.8691, val acc 73.83%

Restored best weights (val acc 75.60%)


## 8. Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [10]:
model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=1e-3)

print("=== Experiment C: Adam + Tanh ===")
history_c = fit(model_c, optimizer_c, train_loader, test_loader)

=== Experiment C: Adam + Tanh ===
Epoch  1/10 | train loss 1.4070, train acc 49.62% | val loss 1.1269, val acc 59.93%
Epoch  2/10 | train loss 1.0132, train acc 64.37% | val loss 0.9440, val acc 66.25%
Epoch  3/10 | train loss 0.8432, train acc 70.43% | val loss 0.8685, val acc 69.61%
Epoch  4/10 | train loss 0.7226, train acc 75.13% | val loss 0.8395, val acc 70.41%
Epoch  5/10 | train loss 0.6256, train acc 78.63% | val loss 0.7803, val acc 73.04%
Epoch  6/10 | train loss 0.5300, train acc 81.95% | val loss 0.7957, val acc 72.89%
Epoch  7/10 | train loss 0.4365, train acc 85.48% | val loss 0.7941, val acc 74.01%
Epoch  8/10 | train loss 0.3545, train acc 88.45% | val loss 0.7942, val acc 73.78%
Epoch  9/10 | train loss 0.2756, train acc 91.71% | val loss 0.8144, val acc 74.16%
Epoch 10/10 | train loss 0.2008, train acc 94.62% | val loss 0.8339, val acc 74.66%

Restored best weights (val acc 74.66%)
